# Issue #10 Quickstart (Standalone)
This notebook demonstrates the new modular sequential pipeline API without changing existing examples.

In [5]:
from pathlib import Path
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

sample_file = Path('../examples/sample_code.py').resolve()

In [2]:
pipeline = GraphPipeline(
    components=[
        JoernParser(granularity=NodeGranularity.LINE),
        KHopTrimmer(hops=1),
        CodeReconstructionSerializer(granularity=NodeGranularity.LINE),
    ]
)

In [7]:
plan = pipeline.dry_run(file_path=str(sample_file), target=1)
plan

[Pipeline] Dry run plan:
  1. JoernParser (file_path -> CodeGraph)
  2. KHopTrimmer (CodeGraph -> CodeGraph)
  3. CodeReconstructionSerializer (CodeGraph -> str)


[{'step': 1,
  'name': 'JoernParser',
  'component_type': 'JoernParser',
  'input_type': 'file_path',
  'output_type': 'CodeGraph',
  'requires_context': ['file_path'],
  'capabilities': ['read_file', 'spawn_process'],
  'granularity': 'LINE',
  'parse_timeout_seconds': 120,
  'export_timeout_seconds': 120},
 {'step': 2,
  'name': 'KHopTrimmer',
  'component_type': 'KHopTrimmer',
  'input_type': 'CodeGraph',
  'output_type': 'CodeGraph',
  'requires_context': ['target_node_id'],
  'capabilities': ['graph_read'],
  'hops': 1},
 {'step': 3,
  'name': 'CodeReconstructionSerializer',
  'component_type': 'CodeReconstructionSerializer',
  'input_type': 'CodeGraph',
  'output_type': 'str',
  'requires_context': [],
  'capabilities': ['write_text'],
  'granularity': 'LINE'}]

In [6]:
result = pipeline.run(file_path=str(sample_file), target=1)
print(result[:500])

[Pipeline] Step 1: running JoernParser on /app/workspace/CodeGraphene/examples/sample_code.py...
[JoernParser] Parsing source code at: /app/workspace/CodeGraphene/examples/sample_code.py
[JoernParser] Running: joern-parse /app/workspace/CodeGraphene/examples/sample_code.py --output /tmp/tmpd98ff033/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpd98ff033/cpg.bin --repr all --out /tmp/tmpd98ff033/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Target resolved to node 30064771072.
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] Step 3: running CodeReconstructionSerializer...
Line 1: callable
Line 6: RET


In [ ]:
source_snippet = """def greet(name):
    return f'hello {name}'
"""

custom_pipeline = GraphPipeline(
    components=[
        JoernParser(granularity=NodeGranularity.LINE),
        KHopTrimmer(hops=1),
        CodeReconstructionSerializer(
            granularity=NodeGranularity.LINE,
            line_template="{line} => {code}",
            separator=" | ",
        ),
    ]
)

snippet_output = custom_pipeline.run(source_code=source_snippet, language='python', target=1)
print(snippet_output)